**Decision Trees**

A decision tree is a supervised learning algorithm used for both classification and regression tasks. It has a hierarchical tree structure consisting of a root node, branches, internal nodes, and leaf nodes. It works like a flowchart that helps in making step-by-step decisions, where:
 1. Internal nodes represent attribute tests
2. Branches represent attribute values
3. Leaf nodes represent final decisions or predictions.

->Parent node: In any two connected nodes, the one which is higher hierarchically is a parent node.

-> Child node: In any two connected nodes, the one which is lower hierarchically is a child node.

-> Root node: The starting node from which the tree starts. It has no child nodes. The root node has no parent node. (dark blue node in the above image)

-> Leaf Node/leaf: Nodes at the end of the tree that do not have any children are leaf nodes, or simply called leaves. (green nodes in the above image)

-> Internal nodes/nodes: All the nodes between the root node and the leaf nodes are internal nodes, or simply called nodes. Internal nodes have both a parent and at least one child. (red nodes in the above image)

-> Splitting: Dividing a node into two or more sub-nodes or adding two or more children to a node.

-> Decision node: When a parent splits into two or more child nodes, then that node is called a decision node.
 Pruning: When we remove the sub-node of a decision node, it is called pruning. Think
of it as the opposite of splitting.
 Branch/Sub-tree: A subsection of the entire tree is called a branch or sub-tree.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load and prepare Titanic data
titanic_train = pd.read_csv("/content/train.csv")   # Read the data

# Input median Age for NA Age values
new_age_var = np.where(titanic_train["Age"].isnull(), # Logical check
                       28,                       # Value if check is true
                       titanic_train["Age"])     # Value if check is false

titanic_train["Age"] = new_age_var

In [ ]:
from sklearn import tree
from sklearn import preprocessing

In [ ]:
# Initialize label encoder
label_encoder = preprocessing.LabelEncoder()

# Convert Sex variable to numeric
encoded_sex = label_encoder.fit_transform(titanic_train["Sex"])

# Initialize model
tree_model = tree.DecisionTreeClassifier()

# Train the model
tree_model.fit(X = pd.DataFrame(encoded_sex),
               y = titanic_train["Survived"])

In [ ]:
import graphviz
#to be created in the view of tree the model created
# Save tree as dot file
dot_data = tree.export_graphviz(tree_model, out_file=None)
graph = graphviz.Source(dot_data)#AP24110010230
graph

The tree's graph show us that it consists of only one decision node that splits the data on the variable sex (the first variable, X[0]). All 314 females end up in one leaf node and all 577 males end up in a different leaf node.

Let's make predictions with this tree and view a table of the results:

In [1]:
# Get survival probability
preds = tree_model.predict_proba(X = pd.DataFrame(encoded_sex))

pd.crosstab(preds[:,0], titanic_train["Sex"])

NameError: name 'tree_model' is not defined

In [ ]:
The table shows that the decision tree managed to create the simple gender-based model where all females survive and all males perish.

Let's create a new decision tree that adds the passenger class variable and see how it changes the resulting predictions:

In [ ]:
# Make data frame of predictors
predictors = pd.DataFrame([encoded_sex, titanic_train["Pclass"]]).T

# Train the model
tree_model.fit(X = predictors,
               y = titanic_train["Survived"])

In [ ]:
# Save tree as dot file
dot_data = tree.export_graphviz(tree_model, out_file=None)
graph = graphviz.Source(dot_data)
graph

In [ ]:
# Get survival probability
preds = tree_model.predict_proba(X = predictors)

# Create a table of predictions by sex and class
pd.crosstab(preds[:,0], columns = [titanic_train["Pclass"],
                                   titanic_train["Sex"]])

Notice that the more complex model still predicts a higher survival rate for women than men, but women in third class only have a 50% predicted death probability while women in first class are predicted to die less than 5% of the time.

The more variables you add to a decision tree, the more yes/no decisions it can make, resulting in a deeper, more complex tree. Adding too much complexity to a decision tree, however, makes it prone to overfitting the training data, which can lead to poor generalization to unseen data. Let's investigate by creating a larger tree with a few more variables:

In [ ]:
predictors = pd.DataFrame([encoded_sex,
                           titanic_train["Pclass"],
                           titanic_train["Age"],
                           titanic_train["Fare"]]).T

# Initialize model with maximum tree depth set to 8
tree_model = tree.DecisionTreeClassifier(max_depth = 8)

tree_model.fit(X = predictors,
               y = titanic_train["Survived"])

In [ ]:
# Save tree as dot file
dot_data = tree.export_graphviz(tree_model, out_file=None)
graph = graphviz.Source(dot_data)
graph

The image above illustrates how complex decision trees can become when you start adding more explanatory variables. You can control the complexity of the tree by altering some of the decision tree function's default parameters. For example, when we made the tree above, we set max_depth = 8, which limited the tree to a depth of 8 (if we hadn't done this the tree would have been much larger!).

For interest's sake, let's check the accuracy of this decision tree model on the training data:

In [ ]:
# Read and prepare test data
titanic_test = pd.read_csv("/content/test.csv")

# Impute missing Age values
new_age_var = np.where(
    titanic_test["Age"].isnull(),
    28,
    titanic_test["Age"]
)

# Impute missing Fare values
new_fare_var = np.where(
    titanic_test["Fare"].isnull(),
    50,
    titanic_test["Fare"]
)

# Update the dataset
titanic_test["Age"] = new_age_var
titanic_test["Fare"] = new_fare_var

# Display the result
print(titanic_test.head())

In [ ]:
# Convert test variables to match model features
encoded_sex_test = label_encoder.fit_transform(titanic_test["Sex"])

test_features = pd.DataFrame([encoded_sex_test,
                              titanic_test["Pclass"],
                              titanic_test["Age"],
                              titanic_test["Fare"]]).T

In [ ]:
# Make test set predictions
test_features = pd.DataFrame(
    test_features,
    columns=["Sex", "Pclass", "Age", "Fare"]
)

test_preds = tree_model.predict(X=test_features)

# Create a submission for Kaggle
submission = pd.DataFrame({
    "PassengerId": titanic_test["PassengerId"],
    "Survived": test_preds
})

# Save submission to CSV
submission.to_csv(
    "tutorial_dectree_submission.csv",
    index=False
)

print(submission.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Select features and target
X = titanic_train[["Sex", "Pclass", "Age", "Fare"]].copy()
y = titanic_train["Survived"]

# Convert categorical Sex values into numeric columns
X = pd.get_dummies(X, columns=["Sex"], dtype=int)

# Handle missing values
X = X.fillna(X.median())

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=12,
    stratify=y
)

# Train the Decision Tree
model = DecisionTreeClassifier(
    max_depth=8,
    random_state=12
)

model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_val)

# Evaluate the model
print("Validation Accuracy:", accuracy_score(y_val, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# Holdout Validation and Cross Validation

When creating a predictive model, we'd like to get an accurate sense of its ability to generalize to unseen data before actually going out and using it on unseen data. As we saw earlier, generating predictions on the training data itself to check the model's accuracy does not work very well: a complex model may fit the training data extremely closely but fail to generalize to new, unseen data. We can get a better sense of a model's expected performance on unseen data by setting a portion of our training data aside when creating a model, and then using that set aside data to evaluate the model's performance. This technique of setting aside some of the training data to assess a model's ability to generalize is known as validation.

Holdout validation and cross validation are two common methods for assessing a model before using it on test data. Holdout validation involves splitting the training data into two parts, a training set and a validation set, building a model with the training set and then assessing performance with the validation set. In theory, model performance on the hold-out validation set should roughly mirror the performance you'd expect to see on unseen test data. In practice, holdout validation is fast and it can work well, especially on large data sets, but it has some pitfalls.

Reserving a portion of the training data for a holdout set means you aren't using all the data at your disposal to build your model in the validation phase. This can lead to suboptimal performance, especially in situations where you don't have much data to work with. In addition, if you use the same holdout validation set to assess too many different models, you may end up finding a model that fits the validation set well due to chance that won't necessarily generalize well to unseen data. Despite these shortcomings, it is worth learning how to use a holdout validation set in Python.

You can create a holdout validation set using the train_test_split() in sklearn's cross_validation library:

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
v_train, v_test = train_test_split(titanic_train,     # Data set to split
                                   test_size = 0.25,  # Split ratio
                                   random_state=1,    # Set random seed
                                   stratify = titanic_train["Survived"]) #*

# Training set size for validation
print(v_train.shape)    #AP24110010230
# Test set size for validation
print(v_test.shape)

*Note: When performing classification, it is desirable for each class in the target variable to have roughly the same proportion across each split of the data. The stratify argument lets you specify a target variable to spread evenly across the train and test splits.*

The output above shows that we successfully created a new training set with roughly 75% of the original data and a validation test set with 25% of the data. We could proceed by building models with this new training set and making predictions on the validation set to assess the models.

Cross validation is a popular alternative to holdout validation that involves splitting the training data into two or more partitions and creating a model for each partition where the partition acts as the validation set and the remaining data acts as the training data. A common form of cross validation is "k-fold" cross validation, which randomly splits data into some number k (a user specified parameter) partitions and then creates k models, each tested against one of the partitions. Each of the k models are then combined into one aggregate final model.

The primary advantage of cross validation is it uses all the training data to build and assess the final model. The main drawback is that building and testing several models can be computationally expensive, so it tends to take much longer than holdout validation. You can create K cross validation splits of the data using the Kfold() function in sklearn's model_selection library:

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=10, shuffle=True, random_state=12)

print(kf.get_n_splits(titanic_train))

In [ ]:
fold_accuracy = []

titanic_train["Sex"] = encoded_sex

for train_fold, valid_fold in kf.split(titanic_train):
    train = titanic_train.loc[train_fold] # Extract train data with cv indices
    valid = titanic_train.loc[valid_fold] # Extract valid data with cv indices

    model = tree_model.fit(X = train[["Sex","Pclass","Age","Fare"]],
                           y = train["Survived"]) #AP24110010230
    valid_acc = model.score(X = valid[["Sex","Pclass","Age","Fare"]],
                            y = valid["Survived"])
    fold_accuracy.append(valid_acc)

print("Accuracy per fold: ", fold_accuracy, "\n")
print("Average accuracy: ", sum(fold_accuracy)/len(fold_accuracy))

Model accuracy can vary significantly from one fold to the next, especially with small data sets, but the average accuracy across the folds gives you an idea of how the model might perform on unseen data.

As with holdout validation, we'd like the target variable's classes to have roughly the same proportion across each fold when performing cross validation for a classification problem. To perform stratified cross validation, use the StratifiedKFold() function instead of KFold().

You use can score a model with stratified cross validation with a single function call with the cross_val_score() function:

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score(estimator= tree_model,     # Model to test
                X= titanic_train[["Sex","Pclass",   # Train Data
                                  "Age","Fare"]],
                y = titanic_train["Survived"],      # Target variable
                scoring = "accuracy",               # Scoring metric
                cv=10)                              # Cross validation folds

print("Accuracy per fold: ")
print(scores)
print("Average accuracy: ", scores.mean())

Notice that the average accuracy across each fold is higher than the non-stratified K-fold example. The cross_val_score function is useful for testing models and tuning model parameters (finding optimal values for arguments like maximum tree depth that affect model performance).

In [ ]:
import matplotlib.pyplot as plt
import numpy
from sklearn import metrics

actual = numpy.random.binomial(1,.9,size = 1000)
predicted = numpy.random.binomial(1,.9,size = 1000)  #AP24110010230

confusion_matrix = metrics.confusion_matrix(actual, predicted)

cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])

cm_display.plot()
plt.show()

The Confusion Matrix created has four different quadrants:

True Negative (Top-Left Quadrant)
False Positive (Top-Right Quadrant)
False Negative (Bottom-Left Quadrant)
True Positive (Bottom-Right Quadrant)

True means that the values were accurately predicted, False means that there was an error or wrong prediction.

Now that we have made a Confusion Matrix, we can calculate different measures to quantify the quality of the model. First, lets look at Accuracy.

In [ ]:
import matplotlib.pyplot as plt
import numpy
from sklearn import metrics

actual = numpy.random.binomial(1,.9,size = 1000)
predicted = numpy.random.binomial(1,.9,size = 1000) #AP24110010230

confusion_matrix = metrics.confusion_matrix(actual, predicted)

cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])

cm_display.plot()
plt.show()

Decision trees are an easily interpretable yet surprisingly expressive form of predictive model. A decision tree of limited depth can provide a good starting point for classification tasks and model complexity is easy adjustable. For our final lesson, we'll learn about random forests, an extension of decision trees that preform very well on a wide range of classification tasks.